# RAG (End to End)

I'm doing this in a separate notebook because uinlike the [other notebook](./rag.ipynb) this will contain just the final product, not lots of iterations.

The idea is the same, create a simple agent that answers questions about chess based on the Laws of Chess from FIDE.

## Setup

### Imports

In [16]:
import re
import json
from pypdf import PdfReader
from collections import OrderedDict
from pathlib import Path
import requests
from constants import *
from litellm import embedding, Message, ModelResponse, completion
from typing import overload, Union
import numpy as np

### Tests

In [6]:
def test_chunking_has_all_articles(laws: dict):
    expected_keys = {
        "1.1", "1.2", "1.3", "2.1", "2.2", "2.3", "2.4", "3.1", "3.2", "3.3", "3.4", "3.5",
        "3.6", "3.7", "3.8", "3.9", "4.1", "4.2", "4.3", "4.4", "4.5", "4.6", "4.7", "5.1",
        "5.2", "6.1", "6.2", "6.3", "6.4", "6.5", "6.6", "6.7", "6.8", "6.9", "6.10", "6.11",
        "6.12", "6.13", "6.14", "7.1", "7.2", "7.3", "7.4", "7.5", "8.1", "8.2", "8.3", "8.4",
        "8.5", "8.6", "8.7", "9.1", "9.2", "9.3", "9.4", "9.5", "9.6", "10.1", "10.2", "11.1",
        "12.1", "12.2", "12.3", "12.4", "12.5", "12.6", "12.7", "12.8", "12.9", "12.10", "13.1",
        "13.2", "13.3", "13.4", "13.5", "13.6", "13.7", "14.1"
    }
    keys = set(laws.keys())
    assert expected_keys.issubset(keys), f"Missing {len(expected_keys - keys)} articles: {expected_keys - keys}"

def test_chunking_has_all_appendices(laws: dict):
    expected_keys = {
        "A.1", "A.2", "A.3", "A.4", "B.1","B.2", "B.3", "C.1", "C.2", "C.3", "C.4", "C.5", "C.6",
        "C.7", "C.8", "C.9", "C.10", "C.11", "C.12", "C.13", "D.1", "E.1", "E.2", "F.1", "F.2", "F.3"
    }
    keys = set(laws.keys())
    assert expected_keys.issubset(keys), f"Missing {len(expected_keys - keys)} appendices: {expected_keys - keys}"

def test_chunking_has_all_sections(laws: dict):
    test_chunking_has_all_articles(laws)
    test_chunking_has_all_appendices(laws)


## Data

### Download

In [8]:


data_folder = Path("../data")

data_folder.mkdir(parents=True, exist_ok=True)

response = requests.get("https://www.fide.com/FIDE/handbook/LawsOfChess.pdf")

if response.ok:
    with open("../data/laws.pdf", "wb") as f:
        f.write(response.content)
else:
    print("Couldn't download PDF.")


### Parsing

In [9]:
reader = PdfReader("../data/laws.pdf")

pages = []
for page in reader.pages[1:]: # skip the first page
    pages.append(page.extract_text())

assert len(pages) == 24

laws_text = ""

for page in pages:
    parsed_page = re.sub(r" \d \n", "", page, count=1)
    laws_text += parsed_page

laws_text[:200]

'FIDE Laws of Chess cover over-the-board play. \nThe English text is the authentic version of the Laws of Chess, which was adopted at the 7 9th FIDE Congress \nat Dresden (Germany), November 2008, coming'

### Chunking

In [10]:
laws_dict = OrderedDict([])

articles_section = re.search(r"BASIC RULES OF PLAY(.*?)APPENDICES", laws_text, flags=re.DOTALL).group(1).strip()
appendices_section = re.search(r"APPENDICES(.*?)Guidelines in case a game needs to be adjourned", laws_text, flags=re.DOTALL).group(1).strip()
adjournment_section = re.search(r"Guidelines in case a game needs to be adjourned(.*)", laws_text, flags=re.DOTALL).group(1).strip()

# ======================== #
#     Chunks Articles      #
# ======================== #
articles = re.split(r"Article \d+:", articles_section)
articles = [article.strip() for article in articles if article.strip()]

assert len(articles) == 14, f"Found {len(articles)}, expected 14."

subarticles = []

for article in articles:
    subs = [sub.strip() for sub in re.split(r"\n\s?(\d+\.\d+)\s", article) if sub.strip()]
    subarticles.extend(subs[1:])

subarticles = [subsection.replace("\n", "") for subsection in subarticles]

for section, text in zip(subarticles[0::2], subarticles[1::2]):
    laws_dict[section] = text

test_chunking_has_all_articles(laws_dict)

# ======================== #
#    Chunks Apprendices    #
# ======================== #
appendices = re.split(r"[A-F]\. ", appendices_section)
appendices = [appendix.strip() for appendix in appendices if appendix.strip()]

assert len(appendices) == 6, f"Found {len(appendices)}, expected 14."

subappendices = []

for appendex in appendices:
    subs = [sub.strip() for sub in re.split(r"\n([A-F]\.\d+) ", appendex) if sub.strip()]
    subappendices.extend(subs[1:])

subappendices = [subappendix.replace("\n", "") for subappendix in subappendices]

for appendix, text in zip(subappendices[0::2], subappendices[1::2]):
    laws_dict[appendix] = text

test_chunking_has_all_appendices(laws_dict)

chunks = []
chunks.extend([f"search_document: {sub}" for sub in subarticles[1::2]])
chunks.extend([f"search_document: {sub}" for sub in subappendices[1::2]])

len(chunks)

104

## Implementation

### Retrieval

In [11]:
def embed_text(chunks):
    embeddings = []
    response = embedding(model=EMBEDDING_MODEL, input=chunks) # embed all at once
    for emb in response.data:
        embeddings.append(np.array(emb["embedding"]))

    return np.array(embeddings)

def cosine_similarity(query, document) -> np.typing.NDArray:
    numerator = query @ document
    denominator = np.linalg.norm(query) * np.linalg.norm(document, axis=0)
    return numerator / denominator

def search(query, document, k = 3) -> list[tuple[int, float]]:
    """Returns the `k` best matches on the `document` for the `query`"""

    response = embedding(model=EMBEDDING_MODEL, input="search_query: " + query)
    prompt_embedding = np.array(response.data[0]["embedding"])

    similarities = cosine_similarity(prompt_embedding, document)

    indexes = np.argpartition(similarities, -k)[-k:]
    indexes = indexes[np.argsort(similarities[indexes])[::-1]]

    return [(index, similarities[index]) for index in indexes]

### Agent

In [53]:
class Agent:
    def __init__(self, model = CHAT_MODEL, document = []):
        self.model = model
        self.document = document
        self.embeddings = embed_text(document)
        self.messages: list[Message] = [
            Message(role="system", content="You're a helpful chess rules assistant. Based on the context given, answer the user question. The questions MUST be answered based on the rules present in the context.")
        ]

    def __make_rag_prompt(self, prompt: str) -> str:
        RAG_PROMPT_TEMPLATE = """Answer the question based on the context below.
If the context doesn't have enough information to answer the questions, say that explicitly.

<context>
{context}
</context>

<question>
{question}
</question>
"""
        sections = "\n".join([self.document[key] for key, _ in search(prompt, self.embeddings.T, k=8)])
        return RAG_PROMPT_TEMPLATE.format(context=sections, question=prompt)

    @overload
    def add_message(self, message: Message) -> None: ...

    @overload
    def add_message(self, content: str, role: str = "user") -> None: ...

    def __rewrite_question(self, prompt: str) -> str:
        REWRITE_PROMPT_TEMPLATE = """
Based on this message history, rewrite the new question to be a standalone question that includes previous context.
Only rewrite the message if it needs additional context from previous messages.
Output ONLY the standalone question.

<message_history>
{history}
</message_history>

<new_question>
{question}
</new_question>

Standalone Question:
"""
        history = ""
        for message in self.messages:
            if message.role == "user":
                history += re.search(r"<question>(.*?)</question>", message.content, flags=re.DOTALL).group(1).strip() + "\n"
            if message.role == "assistant":
                history += message.content + "\n"

        rewrite_template = REWRITE_PROMPT_TEMPLATE.format(
            history=history,
            question=prompt
        )

        response = completion(
            model=self.model,
            messages=[
                Message(role="user", content=rewrite_template)
            ],
        )

        return response.choices[0].message.content

    def add_message(self, message_or_content: Union[Message, str], role: str = "user") -> None:
        if isinstance(message_or_content, Message):
            if message_or_content.role == "user":
                if sum(1 for msg in self.messages if msg.role == "user") >= 1:
                    message_or_content.content = self.__rewrite_question(message_or_content.content)
                message_or_content.content = self.__make_rag_prompt(message_or_content.content)
            self.messages.append(message_or_content)

        elif isinstance(message_or_content, str):
            if sum(1 for msg in self.messages if msg.role == "user") >= 1:
                message_or_content = self.__rewrite_question(message_or_content)
            self.messages.append(Message(role=role, content=self.__make_rag_prompt(message_or_content)))

    def send(self, prompt: str = None):
        if prompt:
            self.add_message(prompt)

        response = completion(
            model=self.model,
            messages=self.messages
        )
        self.__handle_response(response)        

    def __handle_response(self, response: ModelResponse):
        message = response.choices[0].message
        self.add_message(message)

        if message.content:
            print(message.content)

### Test Fire

In [54]:
agent = Agent(document=chunks)

agent.send(prompt="When can a pawn promote?")

A pawn can promote when it reaches the rank furthest from its starting position (the 8th rank for white pawns and the 1st rank for black pawns). Upon reaching this rank, the pawn must be exchanged **as part of the same move** on the same square for a new queen, rook, bishop, or knight of the same color. The choice of piece is finalized when the new piece touches the square of promotion. This promotion is mandatory and immediate. 

This rule is explicitly stated in the context under the promotion section (search_document: "e. When a pawn reaches the rank furthest...").


In [55]:
agent.send(prompt="Do you know any cake recepies?")

The context provided does not contain any information related to cake recipes. Therefore, I cannot answer this question.


In [56]:
agent.send("What was my first question about?")

Your first question was about the conditions under which a pawn can promote in chess. Based on the context provided, the rule states:  
**"When a pawn reaches the rank furthest from its starting position, it must be exchanged as part of the same move on the same square for a new queen, rook, bishop, or knight of the same colour."** (search_document: "e. When a pawn reaches the rank furthest..."). This explicitly describes the promotion condition.


In [57]:
agent.send("And what happens if I arive late to a chess match?")

If you arrive late to a chess match, **you lose the game** immediately, as stated in the context:  
*"Any player who arrives at the chessboard after the start of the session shall lose the game."*  
This rule applies unless the competition's specific rules override this default outcome (e.g., by specifying a different "default time" for late arrivals). However, the context does not provide further details about such exceptions.


In [58]:
print(agent.messages[1].content)
print(agent.messages[-2].content)

Answer the question based on the context below.
If the context doesn't have enough information to answer the questions, say that explicitly.

<context>
search_document: In the case of the promotion of a pa wn, the actual pawn move is indicated, followed immediately by the first letter of the new piece. Examples: d8Q, f8N, b1B, g1R.
search_document: a. The pawn may move forward to the unoccupied square immediately in front of it on the same file, or b. on its first move the pawn may move as in 3.7.a or alternatively it may advance two squares along the same file provided both squares are unoccupied, or c. the pawn may move to a square occupied by an opponent’s piece, which is diagonally in front of it on an adjacent file, capturing that piece.    d. A pawn attacking a square crossed by an opponent’s pawn which has advanced two squares in one move from its original square may capture this opponent’s pawn as though the latter had been moved only one square. This capture is only leg al on 